# `locityper_02_run_batch` — Cloud Batch (dsub)

Run **in a VWB JupyterLab app** after
[`locityper_00_prep_reference.ipynb`](locityper_00_prep_reference.ipynb).
This is the same Nearline `print_reads` minicram → genotype path as the WDL,
submitted with `dsub --provider google-batch` instead of Cromwell.

Cromwell in this workspace has been spotty. Cloud Batch is the more
consistent / cheaper path. The WDL notebook
([`locityper_01_run_stream.ipynb`](locityper_01_run_stream.ipynb)) is
unchanged if you still want it.

Unlike the WDL, genotype is **not** scattered: one VM runs `preproc`, then
GNU parallel over every BED locus, then the summary TSV.

Docs: [`locityper/batch/README.md`](../../locityper/batch/README.md),
[VWB dsub](https://support.workbench.verily.com/docs/guides/workflows/dsub/).

## What this notebook does

1. Check `dsub`, `PET_SA_EMAIL`, `GOOGLE_CLOUD_PROJECT`
2. Write a per-sample CSV (same columns as `configs/batch.header.csv`)
3. Convert it to dsub `--tasks` TSVs (dry-run; always)
4. Optionally submit minicram, then genotype (`--after`)

`SUBMIT_MINICRAM`, `SUBMIT_GENOTYPE`, and `SUBMIT_ALL` stay **off** until
you turn them on. `BED` / `DB_TAR` default to the toy CYP2D6 catalog from
the prep notebook; swap in Isaac’s files before any analysis.

VPC-SC cannot pull Docker Hub (`eichlerlab/locityper:1.4.5.0`) or Broad AR.
Set `PRINT_READS_DOCKER` / `LOCITYPER_DOCKER` to workspace-readable mirrors.

`--stage all` blocks the kernel until every minicram task finishes. Use it
for the one-row smoke CSV only.


In [ ]:
from __future__ import annotations

import csv
import os
import shutil
import subprocess
import sys
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        script = p / "locityper" / "batch" / "submit_dsub.sh"
        if script.is_file():
            return p
        nested = p / "aou-lr-phase-2" / "locityper" / "batch" / "submit_dsub.sh"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError(
        "Cannot find locityper/batch/submit_dsub.sh. Clone "
        "kvg/aou-lr-phase-2 into this app (or cd into the clone) and re-run."
    )


def sh(cmd: list[str], *, check: bool = True, env: dict[str, str] | None = None) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(cmd), flush=True)
    merged = os.environ.copy()
    if env:
        merged.update(env)
    return subprocess.run(cmd, check=check, text=True, env=merged)


REPO_ROOT = find_repo_root()
SUBMIT_SH = REPO_ROOT / "locityper" / "batch" / "submit_dsub.sh"
SCRATCH = Path.cwd() / "locityper_rw_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)

OUTPUT_BUCKET_GS = os.environ.get("LOCITYPER_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources/")
OUT_PREFIX = os.environ.get(
    "LOCITYPER_BATCH_OUT_PREFIX",
    f"{OUTPUT_BUCKET_GS.rstrip('/')}/batchRuns/locityper",
)

SAMPLE_ID = os.environ.get("LOCITYPER_SAMPLE_ID", "1000000")
CRAM = os.environ.get(
    "LOCITYPER_CRAM",
    "gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_1000000.cram",
)
CRAI = os.environ.get(
    "LOCITYPER_CRAI",
    "gs://vwb-aou-datasets-controlled/pooled/wgs/cram/v8_base/wgs_1000000.cram.crai",
)
REF_FA = os.environ.get(
    "LOCITYPER_REF_FA",
    "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta",
)
REF_FAI = os.environ.get(
    "LOCITYPER_REF_FAI",
    "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta.fai",
)
COUNTS_JF = os.environ.get(
    "LOCITYPER_COUNTS_JF",
    "gs://aou-lr-phase2-resources/locityper/refs/counts.Homo_sapiens_assembly38.k25.jf",
)
BED = os.environ.get(
    "LOCITYPER_BED",
    "gs://aou-lr-phase2-resources/locityper/smoke.bed",
)
DB_TAR = os.environ.get(
    "LOCITYPER_DB_TAR",
    "gs://aou-lr-phase2-resources/locityper/vcf_db.smoke.tar.gz",
)
GCLOUD_PROJECT = os.environ.get(
    "LOCITYPER_GCLOUD_PROJECT", os.environ.get("GOOGLE_CLOUD_PROJECT", "")
)

PRINT_READS_DOCKER = os.environ.get(
    "LOCITYPER_PRINT_READS_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-locityper-print-reads:0.1.0",
)
LOCITYPER_DOCKER = os.environ.get("LOCITYPER_DOCKER", "eichlerlab/locityper:1.4.5.0")
TECHNOLOGY = os.environ.get("LOCITYPER_TECHNOLOGY", "illumina")
LOCITYPER_N_CPU = os.environ.get("LOCITYPER_N_CPU", "2")

BATCH_CSV_LOCAL = os.environ.get("LOCITYPER_BATCH_CSV", "")

SUBMIT_MINICRAM = False
SUBMIT_GENOTYPE = False
SUBMIT_ALL = False  # smoke only; blocks until minicram finishes
AFTER_JOB = ""  # minicram job-id for SUBMIT_GENOTYPE
WAIT = False

print("REPO_ROOT:", REPO_ROOT)
print("SUBMIT_SH:", SUBMIT_SH, "exists=" + str(SUBMIT_SH.is_file()))
print("OUTPUT_BUCKET_GS:", OUTPUT_BUCKET_GS)
print("OUT_PREFIX:", OUT_PREFIX)
print("SAMPLE_ID:", SAMPLE_ID or "(unset)")
print("REF_FA:", REF_FA or "(unset)")
print("COUNTS_JF:", COUNTS_JF or "(unset)")
print("BED:", BED or "(unset)")
print("DB_TAR:", DB_TAR or "(unset)")
print("PRINT_READS_DOCKER:", PRINT_READS_DOCKER)
print("LOCITYPER_DOCKER:", LOCITYPER_DOCKER)
print("GCLOUD_PROJECT:", GCLOUD_PROJECT or "(unset)")
print("PET_SA_EMAIL:", os.environ.get("PET_SA_EMAIL", "(unset)"))
print("SUBMIT_MINICRAM:", SUBMIT_MINICRAM, "SUBMIT_GENOTYPE:", SUBMIT_GENOTYPE, "SUBMIT_ALL:", SUBMIT_ALL)
print("dsub:", shutil.which("dsub"))


## `dsub` / PET SA

Cloud Batch jobs run as `PET_SA_EMAIL` on the workspace private VPC (`network` / `subnetwork` in `us-central1`), same flags as the [VWB dsub guide](https://support.workbench.verily.com/docs/guides/workflows/dsub/). In a terminal prefer `dsub_activate`. VWB Jupyter images often ship an OpenSSL / `cryptography` combo that breaks `dsub`; the next cell runs `pip install -U pyOpenSSL cryptography` first. `dsub` is often already installed; `pip install dsub` if it is still missing after that.

dsub copies `--logging` / `--input` with `DSUB_CLOUD_SDK_IMAGE`. Default `gcr.io/.../cloud-sdk:slim` is blocked by VPC-SC; `submit_dsub.sh` sets Google’s pkg.dev `google-cloud-cli:slim`. Logs land at `{out-prefix}/logs/{job-id}.1.log`.


In [ ]:
# dsub fails on stock VWB Jupyter until these two are upgraded.
sh([sys.executable, "-m", "pip", "install", "-U", "pyOpenSSL", "cryptography"])

if shutil.which("dsub") is None:
    sh([sys.executable, "-m", "pip", "install", "-U", "dsub"])
if shutil.which("dsub") is None:
    raise SystemExit(
        "dsub is not on PATH after pip install. Restart the kernel and re-run."
    )
if not os.environ.get("GOOGLE_CLOUD_PROJECT"):
    raise SystemExit("GOOGLE_CLOUD_PROJECT is empty. Open this notebook in a VWB Jupyter app.")
if not os.environ.get("PET_SA_EMAIL"):
    raise SystemExit("PET_SA_EMAIL is empty. VWB sets this in cloud apps.")

sh(["dsub", "--version"], check=False)
print("project:", os.environ["GOOGLE_CLOUD_PROJECT"])
print("PET_SA_EMAIL:", os.environ["PET_SA_EMAIL"])
print("network: projects/{}/global/networks/network".format(os.environ["GOOGLE_CLOUD_PROJECT"]))


## Batch CSV → dsub task tables

Same columns as [`locityper/configs/batch.header.csv`](../../locityper/configs/batch.header.csv). The converter puts the WGS CRAM **and** CRAI in `--env` (not `--input`) so dsub never localizes the requester-pays Nearline objects. This cell always runs; it does not submit.


In [ ]:
csv_path = Path(BATCH_CSV_LOCAL) if BATCH_CSV_LOCAL else SCRATCH / "locityper.batch.csv"
if BATCH_CSV_LOCAL:
    if not csv_path.is_file():
        raise SystemExit(f"BATCH_CSV_LOCAL does not exist: {csv_path}")
    print("using existing CSV:", csv_path)
else:
    required = {
        "SAMPLE_ID": SAMPLE_ID,
        "CRAM": CRAM,
        "CRAI": CRAI,
        "REF_FA": REF_FA,
        "REF_FAI": REF_FAI,
        "COUNTS_JF": COUNTS_JF,
        "BED": BED,
        "DB_TAR": DB_TAR,
    }
    missing = [k for k, v in required.items() if not v]
    if missing:
        raise SystemExit(f"fill {missing} in the config cell (or set BATCH_CSV_LOCAL)")
    fieldnames = [
        "sample_id",
        "cram",
        "crai",
        "ref_fa_uncompressed",
        "ref_fai_uncompressed",
        "counts_jf",
        "bed",
        "locityper_db_tar_gz",
        "gcloud_project",
    ]
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        w.writeheader()
        w.writerow(
            {
                "sample_id": SAMPLE_ID,
                "cram": CRAM,
                "crai": CRAI,
                "ref_fa_uncompressed": REF_FA,
                "ref_fai_uncompressed": REF_FAI,
                "counts_jf": COUNTS_JF,
                "bed": BED,
                "locityper_db_tar_gz": DB_TAR,
                "gcloud_project": GCLOUD_PROJECT,
            }
        )
    print("wrote one-row CSV:", csv_path)

print(csv_path.read_text())

tasks_dir = SCRATCH / "dsub_tasks"
sh(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "csv_to_dsub_tasks.py"),
        "--pipeline",
        "locityper",
        "--csv",
        str(csv_path),
        "--out-prefix",
        OUT_PREFIX,
        "--outdir",
        str(tasks_dir),
    ]
)
print("--- minicram.tasks.tsv ---")
print((tasks_dir / "minicram.tasks.tsv").read_text())
print("--- genotype.tasks.tsv (CRAM is not an --input) ---")
print((tasks_dir / "genotype.tasks.tsv").read_text())


## Submit

Turn on **one** flag in the config cell:

- `SUBMIT_MINICRAM` — Nearline `print_reads` extract (not preemptible)
- `SUBMIT_GENOTYPE` — requires `AFTER_JOB` = minicram job-id
- `SUBMIT_ALL` — smoke only; waits on minicram via `--after`


In [ ]:
n_flags = int(SUBMIT_MINICRAM) + int(SUBMIT_GENOTYPE) + int(SUBMIT_ALL)
if n_flags == 0:
    print("SUBMIT_* flags are off; not submitting. Flip one in the config cell and re-run.")
elif n_flags > 1:
    raise SystemExit("set only one of SUBMIT_MINICRAM, SUBMIT_GENOTYPE, SUBMIT_ALL")
else:
    env = {
        "OUTPUT_BUCKET_GS": OUTPUT_BUCKET_GS,
        "PRINT_READS_DOCKER": PRINT_READS_DOCKER,
        "LOCITYPER_DOCKER": LOCITYPER_DOCKER,
        "TECHNOLOGY": TECHNOLOGY,
        "LOCITYPER_N_CPU": str(LOCITYPER_N_CPU),
    }
    cmd = [str(SUBMIT_SH), "--csv", str(csv_path), "--out-prefix", OUT_PREFIX]
    if SUBMIT_MINICRAM:
        cmd += ["--stage", "minicram"]
    elif SUBMIT_GENOTYPE:
        if not AFTER_JOB:
            raise SystemExit("SUBMIT_GENOTYPE needs AFTER_JOB = the minicram job-id")
        cmd += ["--stage", "genotype", "--after", AFTER_JOB]
    else:
        cmd += ["--stage", "all"]
    if WAIT:
        cmd.append("--wait")
    sh(cmd, env=env)
    print("job-id files:", list((REPO_ROOT / "locityper" / "batch" / ".tasks").glob("*.job_id")))


## Where outputs go


In [ ]:
# After submit, poll with dstat (replace JOB_ID):
#   dstat --provider google-batch --project "$GOOGLE_CLOUD_PROJECT" --jobs JOB_ID --status '*'
print("OUT_PREFIX:", OUT_PREFIX)
print("example outputs:")
print(f"  {OUT_PREFIX.rstrip('/')}/{SAMPLE_ID}/{SAMPLE_ID}.minicram.cram")
print(f"  {OUT_PREFIX.rstrip('/')}/{SAMPLE_ID}/{SAMPLE_ID}.gts.filtered.csv")
print(f"  {OUT_PREFIX.rstrip('/')}/{SAMPLE_ID}/{SAMPLE_ID}.locityper.tar.gz")
print("logs:", f"{OUT_PREFIX.rstrip('/')}/logs")
